# Full-Scale TruthfulQA Runner (Updated)

This notebook includes everything required: setup, Ollama install/start, model pulls, full-study run, evaluation, plots, publication artifacts, and export.

- Set `RUN_STRONG_MACHINE = True` for **all 3 models** (`llama3.2`, `mistral`, `llama3.3:70b`).
- Set `RUN_STRONG_MACHINE = False` for only 2 models (`llama3.2`, `mistral`).

In [ ]:
!git clone https://github.com/sahelmain/llm-hallucination-phoenix.git
%cd llm-hallucination-phoenix
!pip install -q -r requirements.txt

In [ ]:
# Install and start Ollama
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
import time

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
!ollama --version
!ollama list

In [ ]:
# Choose mode here
RUN_STRONG_MACHINE = True
print("RUN_STRONG_MACHINE =", RUN_STRONG_MACHINE)

In [ ]:
# Pull models based on machine strength
if RUN_STRONG_MACHINE:
    !ollama pull llama3.2
    !ollama pull mistral
    !ollama pull llama3.3:70b
else:
    !ollama pull llama3.2
    !ollama pull mistral

!ollama list

In [ ]:
# Update config to match selected mode (models + templates)
import yaml

with open("config/experiment.yaml") as f:
    cfg = yaml.safe_load(f)

if RUN_STRONG_MACHINE:
    cfg["models"]["full_study_models"] = [
        {"provider": "ollama", "model_name": "llama3.2", "temperature": 0.0, "max_tokens": 128},
        {"provider": "ollama", "model_name": "mistral", "temperature": 0.0, "max_tokens": 128},
        {"provider": "ollama", "model_name": "llama3.3:70b", "temperature": 0.0, "max_tokens": 128},
    ]
else:
    cfg["models"]["full_study_models"] = [
        {"provider": "ollama", "model_name": "llama3.2", "temperature": 0.0, "max_tokens": 128},
        {"provider": "ollama", "model_name": "mistral", "temperature": 0.0, "max_tokens": 128},
    ]

# Always run the expanded 4-template full study matrix
cfg["prompt_templates"]["full_study"] = [
    "factual_direct",
    "strict_abstain",
    "chain_of_thought",
    "concise_factual",
]

with open("config/experiment.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Configured models:", [m["model_name"] for m in cfg["models"]["full_study_models"]])
print("Configured full-study templates:", cfg["prompt_templates"]["full_study"])

In [ ]:
# Optional clean rerun: remove old full-study outputs before running.
# Set to False if you want resume behavior.
CLEAN_RERUN = True

if CLEAN_RERUN:
    !rm -f data/full_study_results.csv \
          data/evaluated_results_full.csv \
          data/metrics_full.json \
          data/category_metrics_full.csv \
          data/category_why_signals_full.csv \
          data/paired_comparisons_full.json \
          data/paired_comparisons_by_category_full.json \
          data/table_top10_vulnerable_categories_full.csv \
          data/table_model_category_hallucination_full.csv \
          data/table_top_failure_reason_by_model_category_full.csv \
          reports/full_reproducibility_appendix.md
    print("Old full-study outputs removed. Fresh rerun enabled.")
else:
    print("Keeping existing outputs. Resume mode enabled.")

In [ ]:
# Long-running generation. Rerun safely (resume/checkpoint supported).
!python src/run_round2_matrix.py --mode full --disable-phoenix

In [ ]:
# Evaluate deterministic non-LLM metrics + category analyses
!python src/evaluate_metrics.py

In [ ]:
# Generate plots and publication artifacts
!python src/generate_plots.py
!python src/generate_full_study_artifacts.py --suffix full

In [ ]:
# Validate run coverage
import pandas as pd
df = pd.read_csv("data/full_study_results.csv")
n_items = df["item_id"].nunique()
n_models = df["model"].nunique()
n_templates = df["template"].nunique()
n_prompt_types = df["prompt_type"].nunique()
n_reps = df["repetition"].nunique()
expected = n_items * n_models * n_templates * n_prompt_types * n_reps
print("rows:", len(df))
print("expected:", expected)
print("models:", sorted(df['model'].unique().tolist()))
print("errors:", int(df['output'].astype(str).str.startswith('[ERROR]').sum()))

In [ ]:
# Download outputs bundle
!zip -r full_study_outputs.zip data/ reports/
from google.colab import files
files.download("full_study_outputs.zip")

## Optional — Push results to GitHub

Add `GH_TOKEN` to Colab Secrets first (key icon in left sidebar), then run:

```python
from google.colab import userdata
token = userdata.get('GH_TOKEN')
!git add -A
!git config user.email 'sahelmain@example.com'
!git config user.name 'Sahel Azzam'
!git commit -m 'Add full-study results from Colab' || true
!git remote set-url origin https://{token}@github.com/sahelmain/llm-hallucination-phoenix.git
!git push origin main
```